In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_33_try_0.pickle

In [ ]:
%%PandasProfile
### cell 34 ###

# Optimized cell 34

def grab_subset_of_data_46(df, question):
    # Select, rename and drop all‐NA rows in one chain
    return (
        df.filter(like=question)
          .rename(columns=lambda x: x.rsplit('-', 1)[-1].lstrip())
          .dropna(how='all')
    )


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_46(
    question, include_2017=False
):
    # Map years to their DataFrames
    mapping = {
        '2022': responses_df_2022_cell10,
        '2021': responses_df_2021,
        '2020': responses_df_2020,
        '2019': responses_df_2019_cell10,
        '2018': responses_df_2018_cell10,
        '2017': responses_df_2017
    }
    years = ['2018', '2019', '2020', '2021', '2022']
    if include_2017:
        years.insert(0, '2017')
    # Grab, tag and concat in one go
    dfs = [
        grab_subset_of_data_46(mapping[y], question).assign(year=y)
        for y in years
    ]
    return pd.concat(dfs, ignore_index=True)

# Rename 2018 question to match
old = 'What machine learning frameworks have you used in the past 5 years?'
new = 'Which of the following machine learning frameworks do you use on a regular basis?'
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
        .str.replace(old, new, regex=False)
)

question = new
# Combine 2018–2022 (2017 excluded)
ml_df_combined = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_46(
        question
    )
)

# Define groups of equivalent columns
col_groups = {
    'TensorFlow ': ['TensorFlow', 'TensorFlow '],
    'Keras ':      ['Keras',      'Keras '],
    'PyTorch ':    ['PyTorch',    'PyTorch '],
    'Scikit-learn ': [c for c in ml_df_combined.columns if c.lower().startswith('scikit')]
}

# Precompute presence of all old cols once
all_old = [c for olds in col_groups.values() for c in olds if c in ml_df_combined.columns]
presence = ml_df_combined[all_old].notna() if all_old else None

# Build the new flag columns
flags = {}
for new_col, olds in col_groups.items():
    existing = [c for c in olds if c in ml_df_combined.columns]
    if existing:
        flags[new_col] = np.where(
            presence[existing].any(axis=1), new_col, np.nan
        )

# Apply flags and drop old variants
ml_df_combined_cell46 = (
    ml_df_combined
      .assign(**flags)
      .drop(columns=all_old)
)

# Compute counts and convert to percentages
counts = ml_df_combined_cell46.groupby('year').count()
totals = ml_df_combined_cell46['year'].value_counts().sort_index()
ml_df_combined_percentages = (
    counts.div(totals, axis=0) * 100
).reset_index()

# Ensure all expected columns
expected = ['Scikit-learn ', 'TensorFlow ', 'Keras ', 'PyTorch ', 'None', 'Other']
ml_df_combined_percentages = ml_df_combined_percentages.reindex(
    columns=['year'] + expected,
    fill_value=0
)

# Melt for plotting/table
df_cell46 = (
    ml_df_combined_percentages
      .melt(id_vars=['year'], value_vars=expected)
      .sort_values(['year', 'value'], ascending=True)
      .rename(columns={'variable': ''})
)

df_cell46.info()